In [1]:
import json
import torch
import numpy as np
from pathlib import Path
from tqdm import tqdm

from implicit_reps import *
from bns_utils import stringify_config

In [2]:
import numpy as np
from skimage import measure
import trimesh

# --- Device setup ---
device = 'mps'
print(f"Using device: {device}")

# --- Load config, same pattern as Surface-Fitter-fast.ipynb ---
config_filepath = "configs/surfaces/arm500-inv-exp.json"
#config_filepath = "configs/surfaces/cat-reference500.json"

with open(config_filepath, "r") as f:
    config_dict = json.load(f)
    namestring = stringify_config(config_dict)
    print('Namestring will be ', namestring)

    shape_name = config_dict['shape-name']
    surface_config = config_dict['surface-config']
    training_config = config_dict['training-config']

posenc = training_config.get('posenc', True)

if 'sdf_weights_path' in training_config.keys():
    DEEPSDF_MODEL = load_deepsdf_model(
        ckpt_path='sdf_weights/surfaces/' + training_config['sdf_weights_path'],
        device=device, posenc=posenc
    )
else:
    DEEPSDF_MODEL = None

if 'sdf_transition_width' in training_config.keys():
    transition_width = training_config['sdf_transition_width']
else:
    transition_width = 0.0

sdf_id = training_config['sdf_id']

# --- Evaluate the SDF on a regular grid ---
res = 64  # you can increase for finer detail
lin = torch.linspace(-1, 1, res, device=device)
grid_x, grid_y, grid_z = torch.meshgrid(lin, lin, lin, indexing='ij')
grid_pts = torch.stack([grid_x, grid_y, grid_z], dim=-1).reshape(1, -1, 3)

with torch.no_grad():
    sdf_vals = sdf(
        grid_pts, sdf_id,
        squared=False, model=DEEPSDF_MODEL, transition_width=transition_width
    )
sdf_vals = sdf_vals.reshape(res, res, res).cpu().numpy()

# --- Marching cubes ---
# Note: `level=0` extracts the zero level set (surface)
verts, faces, normals, values = measure.marching_cubes(sdf_vals, level=0.0)

# Scale vertices from grid coordinates [-1,1] space
verts = verts / res * 2.0 - 1.0

# --- Visualize / export ---
mesh = trimesh.Trimesh(vertices=verts, faces=faces, vertex_normals=normals)
mesh.export("data/output_mesh.ply")
print("Mesh saved to output_mesh.ply")
mesh.show()

Using device: mps
Namestring will be  arm___surf__=pou_inv_exp__=arm500__deg=2__overlap=0.73__global-scale=0.5__local-scales=T___train__num-per-face=10000__maxepochs=20__b-size=100__initlr=0.001__minlr=1e-05__Nreg=0__Dreg=0__AreaW=F
Loading deepsdf model onto  mps


/Users/romywilliamson/miniconda3/envs/bens/lib/python3.8/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


DeepSDF Model loaded successfully on device: mps
Mesh saved to output_mesh.ply
